# ESM2 Ubiquitination Prediction (Clean Pipeline)

This notebook builds a clean, reproducible protein-level classifier to predict whether a protein is ubiquitinated, using **ESM2** (1280-dim) pooled embeddings.

## What this notebook does
- Loads UniProt table and defines binary labels (ubiquitinated vs not ubiquitinated).
- Loads precomputed ESM2 embeddings (max pool and attention pool, 1280-dim).
- Trains and compares Logistic Regression and MLP models.
- Evaluates ROC-AUC, PR-AUC, precision, recall, F1, and confusion matrices.
- Saves clean plots and summary tables to `results_ubiquitination_ESM2/`.

In [ ]:
from pathlib import Path
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
)

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 140

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
PROJECT_DIR = Path.cwd()
RESULTS_DIR = PROJECT_DIR / "results_ubiquitination_ESM2"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

candidate_data_dirs = [
    (PROJECT_DIR / ".." / "all_uniref").resolve(),
    (PROJECT_DIR / ".." / ".." / "all_uniref").resolve(),
    Path("/home/saishyam/Protein_dynamics/all_uniref"),
    Path("/nfs/turbo/umms-mcieslik/saishyam/Protein_dynamics/all_uniref"),
]
DATA_DIR = next((p for p in candidate_data_dirs if p.exists()), candidate_data_dirs[0])
UNIPROT_FILE = DATA_DIR / "uniprotkb_AND_reviewed_true_AND_model_o_2026_01_16.tsv"

candidate_embed_files = [
    DATA_DIR / "esm2_embedding_cache" / "pooled_embeddings.npz",
    Path("/nfs/turbo/umms-mcieslik/saishyam/Protein_dynamics/all_uniref/esm2_embedding_cache/pooled_embeddings.npz"),
]
EMBED_FILE = next((p for p in candidate_embed_files if p.exists()), candidate_embed_files[0])

print(f"Project directory : {PROJECT_DIR}")
print(f"Data directory    : {DATA_DIR}")
print(f"Results directory : {RESULTS_DIR}")
print(f"UniProt file      : {UNIPROT_FILE}  (exists: {UNIPROT_FILE.exists()})")
print(f"Embedding file    : {EMBED_FILE}  (exists: {EMBED_FILE.exists()})")

In [ ]:
all_uniref = pd.read_csv(UNIPROT_FILE, sep="\t")
print(f"Loaded {len(all_uniref):,} proteins")
all_uniref.head()

In [ ]:
def build_clean_label_sets(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    ubi_mask = (
        df["Post-translational modification"]
        .fillna("")
        .str.contains("ubiquitin", case=False, regex=True)
    )
    ubi  = df[ubi_mask].copy()
    nubi = df[~ubi_mask].copy()

    cols = ["Entry", "Protein names", "Sequence"]
    rename_map = {
        "Entry": "protein_id",
        "Protein names": "protein_name",
        "Sequence": "sequence",
    }

    ubi  = ubi.loc[:, cols].rename(columns=rename_map)
    nubi = nubi.loc[:, cols].rename(columns=rename_map)

    ubi  = ubi.drop_duplicates(subset="protein_name", keep="first")
    nubi = nubi.drop_duplicates(subset="protein_name", keep="first")

    overlap = set(ubi["protein_name"]) & set(nubi["protein_name"])
    ubi  = ubi[~ubi["protein_name"].isin(overlap)]
    nubi = nubi[~nubi["protein_name"].isin(overlap)]

    ubi  = ubi.reset_index(drop=True)
    nubi = nubi.reset_index(drop=True)

    assert ubi["protein_name"].is_unique
    assert nubi["protein_name"].is_unique
    assert set(ubi["protein_name"]).isdisjoint(set(nubi["protein_name"]))

    return ubi, nubi

ubi_df, nubi_df = build_clean_label_sets(all_uniref)

class_counts = pd.DataFrame({
    "Class": ["Ubiquitinated", "Not ubiquitinated"],
    "Count": [len(ubi_df), len(nubi_df)]
})
class_counts["Fraction"] = class_counts["Count"] / class_counts["Count"].sum()
print(class_counts.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.barplot(data=class_counts, x="Class", y="Count", hue="Class",
            palette=["#d62728", "#1f77b4"], ax=ax, legend=False)
ax.set_title("Class Distribution — ESM2 Ubiquitination")
ax.set_xlabel("")
ax.set_ylabel("Number of proteins")
for i, row in class_counts.iterrows():
    ax.text(i, row["Count"] * 1.01, f"{row['Count']} ({row['Fraction']:.1%})",
            ha="center", va="bottom", fontsize=11)
plt.tight_layout()
class_dist_path = RESULTS_DIR / "class_distribution.png"
plt.savefig(class_dist_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {class_dist_path}")

In [ ]:
print(f"Loading ESM2 embeddings from: {EMBED_FILE}")
t0 = time.perf_counter()
cache     = np.load(EMBED_FILE, allow_pickle=True)
max_pool  = cache["max"].item()
attn_pool = cache["attn"].item()
print(f"Loaded in {time.perf_counter()-t0:.1f}s  ({len(max_pool):,} proteins)")

def collect_embeddings(df: pd.DataFrame, pool_dict: dict, label: str) -> np.ndarray:
    vectors = []
    missing = []
    for pid in df["protein_id"]:
        vec = pool_dict.get(pid)
        if vec is None:
            missing.append(pid)
        else:
            vectors.append(vec)
    if not vectors:
        raise ValueError(f"No embeddings found for class: {label}")
    if missing:
        print(f"[{label}] Missing embeddings: {len(missing)}")
    return np.stack(vectors)

X_ubi_max   = collect_embeddings(ubi_df,  max_pool,  "ubiquitinated|max")
X_nubi_max  = collect_embeddings(nubi_df, max_pool,  "not_ubiquitinated|max")
X_ubi_attn  = collect_embeddings(ubi_df,  attn_pool, "ubiquitinated|attn")
X_nubi_attn = collect_embeddings(nubi_df, attn_pool, "not_ubiquitinated|attn")

print("Max pool shapes :", X_ubi_max.shape, X_nubi_max.shape)
print("Attn pool shapes:", X_ubi_attn.shape, X_nubi_attn.shape)
assert X_ubi_max.shape[1] == 1280, f"Expected 1280-dim ESM2 embeddings, got {X_ubi_max.shape[1]}"

In [ ]:
def make_split(pos: np.ndarray, neg: np.ndarray):
    X = np.vstack([pos, neg]).astype(np.float32)
    y = np.array([1] * len(pos) + [0] * len(neg), dtype=np.int64)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=SEED
    )
    scaler  = StandardScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_test  = scaler.transform(X_test).astype(np.float32)
    return X_train, X_test, y_train, y_test

split_data = {
    "max":  make_split(X_ubi_max,  X_nubi_max),
    "attn": make_split(X_ubi_attn, X_nubi_attn),
}

def evaluate_binary(y_true: np.ndarray, y_prob: np.ndarray, threshold: float = 0.5) -> dict:
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "roc_auc":   roc_auc_score(y_true, y_prob),
        "pr_auc":    average_precision_score(y_true, y_prob),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall":    recall_score(y_true, y_pred, zero_division=0),
        "f1":        f1_score(y_true, y_pred, zero_division=0),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

results    = []
curve_data = {}

t0 = time.perf_counter()
for pooling_name, (Xtr, Xte, ytr, yte) in split_data.items():
    clf = LogisticRegression(
        max_iter=5000, class_weight="balanced", random_state=SEED
    )
    clf.fit(Xtr, ytr)
    prob = clf.predict_proba(Xte)[:, 1]

    metrics = evaluate_binary(yte, prob)
    metrics.update({
        "model": "logistic_regression", "pooling": pooling_name,
        "n_test": int(len(yte)), "positive_rate_test": float(yte.mean()),
    })
    results.append(metrics)

    fpr, tpr, _  = roc_curve(yte, prob)
    prec, rec, _ = precision_recall_curve(yte, prob)
    curve_data[("logistic_regression", pooling_name)] = {
        "y_true": yte, "y_prob": prob,
        "fpr": fpr, "tpr": tpr, "precision": prec, "recall": rec,
    }

print(f"Logistic regression done ({time.perf_counter()-t0:.1f}s)")

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(256, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


def train_mlp(Xtr, ytr, Xte, yte, epochs=30, batch_size=64, lr=1e-3):
    Xtr_t = torch.from_numpy(Xtr)
    ytr_t = torch.from_numpy(ytr.astype(np.float32))
    Xte_t = torch.from_numpy(Xte)

    train_loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=batch_size, shuffle=True)

    pos = ytr.sum()
    neg = len(ytr) - pos
    pos_weight = torch.tensor([neg / max(pos, 1)], dtype=torch.float32, device=device)

    model     = MLP(Xtr.shape[1]).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    losses = []
    model.train()
    for _ in range(epochs):
        running = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            running += loss.item()
        losses.append(running / len(train_loader))

    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(Xte_t.to(device))).cpu().numpy()

    return probs, losses


training_loss = {}
t0 = time.perf_counter()
for pooling_name, (Xtr, Xte, ytr, yte) in split_data.items():
    probs, losses = train_mlp(Xtr, ytr, Xte, yte)
    metrics = evaluate_binary(yte, probs)
    metrics.update({
        "model": "mlp", "pooling": pooling_name,
        "n_test": int(len(yte)), "positive_rate_test": float(yte.mean()),
    })
    results.append(metrics)

    fpr, tpr, _  = roc_curve(yte, probs)
    prec, rec, _ = precision_recall_curve(yte, probs)
    curve_data[("mlp", pooling_name)] = {
        "y_true": yte, "y_prob": probs,
        "fpr": fpr, "tpr": tpr, "precision": prec, "recall": rec,
    }
    training_loss[pooling_name] = losses

print(f"MLP done ({time.perf_counter()-t0:.1f}s)")

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df[[
    "model", "pooling", "roc_auc", "pr_auc",
    "precision", "recall", "f1",
    "tp", "fp", "tn", "fn",
    "n_test", "positive_rate_test",
]]
results_df = results_df.sort_values(["pr_auc", "roc_auc"], ascending=False).reset_index(drop=True)

csv_path = RESULTS_DIR / "summary_metrics.csv"
md_path  = RESULTS_DIR / "summary_metrics.md"
results_df.to_csv(csv_path, index=False)
header    = "| " + " | ".join(results_df.columns) + " |"
separator = "|" + "|".join(["---"] * len(results_df.columns)) + "|"
rows_md   = ["| " + " | ".join(map(str, row)) + " |" for row in results_df.to_numpy()]
md_path.write_text("\n".join([header, separator] + rows_md))

display(results_df.style.format({
    "roc_auc": "{:.4f}", "pr_auc": "{:.4f}",
    "precision": "{:.4f}", "recall": "{:.4f}",
    "f1": "{:.4f}", "positive_rate_test": "{:.4f}",
}))
print(f"Saved: {csv_path}")
print(f"Saved: {md_path}")

In [ ]:
palette = {
    ("logistic_regression", "max"):  "#1f77b4",
    ("logistic_regression", "attn"): "#ff7f0e",
    ("mlp", "max"):                  "#2ca02c",
    ("mlp", "attn"):                 "#d62728",
}

# ROC curves
fig, ax = plt.subplots(figsize=(8.5, 6))
for key, data in curve_data.items():
    model, pooling = key
    label = f"{model} | {pooling} (AUC={roc_auc_score(data['y_true'], data['y_prob']):.3f})"
    ax.plot(data["fpr"], data["tpr"], label=label, color=palette[key], linewidth=2)
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1.5)
ax.set_title("ROC Curves — ESM2 Ubiquitination")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend(fontsize=10, loc="lower right")
plt.tight_layout()
roc_path = RESULTS_DIR / "roc_curves.png"
plt.savefig(roc_path, dpi=300, bbox_inches="tight")
plt.show()

# Precision-Recall curves
fig, ax = plt.subplots(figsize=(8.5, 6))
baseline = list(curve_data.values())[0]["y_true"].mean()
for key, data in curve_data.items():
    model, pooling = key
    label = f"{model} | {pooling} (AP={average_precision_score(data['y_true'], data['y_prob']):.3f})"
    ax.plot(data["recall"], data["precision"], label=label, color=palette[key], linewidth=2)
ax.axhline(baseline, linestyle="--", color="gray", linewidth=1.5,
           label=f"Random baseline={baseline:.3f}")
ax.set_title("Precision-Recall Curves — ESM2 Ubiquitination")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.legend(fontsize=10, loc="upper right")
plt.tight_layout()
pr_path = RESULTS_DIR / "pr_curves.png"
plt.savefig(pr_path, dpi=300, bbox_inches="tight")
plt.show()

# Confusion matrices at threshold 0.5
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for ax, key in zip(axes.ravel(), curve_data.keys()):
    model, pooling = key
    y_true = curve_data[key]["y_true"]
    y_pred = (curve_data[key]["y_prob"] >= 0.5).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["Non-Ubi", "Ubi"], yticklabels=["Non-Ubi", "Ubi"])
    ax.set_title(f"{model} | {pooling}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
cm_path = RESULTS_DIR / "confusion_matrices.png"
plt.savefig(cm_path, dpi=300, bbox_inches="tight")
plt.show()

# MLP training loss
fig, ax = plt.subplots(figsize=(8.5, 6))
ax.plot(training_loss["max"],  label="MLP | max",  linewidth=2)
ax.plot(training_loss["attn"], label="MLP | attn", linewidth=2)
ax.set_title("MLP Training Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("BCEWithLogitsLoss")
ax.legend()
plt.tight_layout()
loss_path = RESULTS_DIR / "mlp_training_loss.png"
plt.savefig(loss_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved: {roc_path}")
print(f"Saved: {pr_path}")
print(f"Saved: {cm_path}")
print(f"Saved: {loss_path}")

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

X_vis = np.vstack([X_ubi_attn, X_nubi_attn])
y_vis = np.array(["Ubiquitinated"] * len(X_ubi_attn) + ["Not ubiquitinated"] * len(X_nubi_attn))

pca_2d = PCA(n_components=2, random_state=SEED).fit_transform(X_vis)

# Stratified subsample for t-SNE (fast)
max_tsne_points = 10000
if len(X_vis) > max_tsne_points:
    rng = np.random.default_rng(SEED)
    idx_pos = np.where(y_vis == "Ubiquitinated")[0]
    idx_neg = np.where(y_vis == "Not ubiquitinated")[0]
    n_pos = max(1, int(max_tsne_points * len(idx_pos) / len(X_vis)))
    n_neg = max_tsne_points - n_pos
    sample_idx = np.concatenate([
        rng.choice(idx_pos, size=min(n_pos, len(idx_pos)), replace=False),
        rng.choice(idx_neg, size=min(n_neg, len(idx_neg)), replace=False),
    ])
    X_tsne_in = X_vis[sample_idx]
    y_tsne    = y_vis[sample_idx]
    print(f"t-SNE on stratified subset: {len(sample_idx):,} points")
else:
    X_tsne_in = X_vis
    y_tsne    = y_vis
    print(f"t-SNE on full set: {len(X_vis):,} points")

t0 = time.perf_counter()
tsne_2d = TSNE(n_components=2, init="pca", learning_rate="auto",
               perplexity=30, random_state=SEED).fit_transform(X_tsne_in)
print(f"t-SNE complete in {time.perf_counter()-t0:.1f}s")

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, emb, y_plot, title in [
    (axes[0], pca_2d,   y_vis,   "PCA (ESM2 Attention embeddings)"),
    (axes[1], tsne_2d,  y_tsne,  "t-SNE (ESM2 Attention embeddings)"),
]:
    idx_neg = y_plot == "Not ubiquitinated"
    idx_pos = y_plot == "Ubiquitinated"
    ax.scatter(emb[idx_neg, 0], emb[idx_neg, 1], s=14, alpha=0.35, c="#1f77b4", label="Not ubiquitinated")
    ax.scatter(emb[idx_pos, 0], emb[idx_pos, 1], s=14, alpha=0.55, c="#d62728", label="Ubiquitinated")
    ax.set_title(title)
    ax.set_xlabel("Dim 1")
    ax.set_ylabel("Dim 2")
axes[0].legend(loc="best")
plt.tight_layout()
emb_path = RESULTS_DIR / "embedding_projection_pca_tsne.png"
plt.savefig(emb_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {emb_path}")

In [ ]:
best_row = results_df.iloc[0]
summary_lines = [
    "ANALYSIS SUMMARY — ESM2 Ubiquitination",
    "========================================",
    f"Best model by PR-AUC: {best_row['model']} + {best_row['pooling']}",
    f"ROC-AUC  : {best_row['roc_auc']:.4f}",
    f"PR-AUC   : {best_row['pr_auc']:.4f}",
    f"Precision: {best_row['precision']:.4f}",
    f"Recall   : {best_row['recall']:.4f}",
    f"F1       : {best_row['f1']:.4f}",
    "",
    "Dataset:",
    f"  Ubiquitinated proteins     : {len(ubi_df)}",
    f"  Non-ubiquitinated proteins : {len(nubi_df)}",
    f"  Positive rate (test set)   : {best_row['positive_rate_test']:.4f}",
    "",
    "Interpretation:",
    "- PR-AUC is emphasised because classes are imbalanced (~7% positive).",
    "- ESM2 uses 1280-dim embeddings (vs 768 for Ankh).",
    "- Attention pooling generally improves PR-AUC over max pooling.",
    "- Confusion matrices show threshold-0.5 operating behaviour.",
]

analysis_path = RESULTS_DIR / "analysis_summary.txt"
analysis_path.write_text("\n".join(summary_lines))
print("\n".join(summary_lines))
print(f"\nSaved: {analysis_path}")